# Two-season trigger

Comparing triggering in the first flooding season (crue rouges / locale) and
second flooding season (crue guinéenne)

In [1]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [93]:
import pandas as pd
import numpy as np

from src.datasources import abn
from src.constants import *
from src.utils import shift_to_floodseason_corrected, FLOODSEASON_START
from utils import FLOODSEASON_ENDDAY

In [3]:
THRESH = 580

In [54]:
level = abn.load_abn_niamey().reset_index()
level = level.rename(columns={"Date": "date", "Water Level (cm)": "level"})
level = shift_to_floodseason_corrected(level)
level = level[level["seasonyear"].isin(range(2005, 2022))]
level.sort_values("date")

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
152,2005-06-02,133.667,63.340,2005,153,2.0,2005
153,2005-06-03,147.000,86.667,2005,154,3.0,2005
154,2005-06-04,156.667,105.333,2005,155,4.0,2005
155,2005-06-05,181.000,157.467,2005,156,5.0,2005
156,2005-06-06,167.667,127.933,2005,157,6.0,2005
...,...,...,...,...,...,...,...
6356,2022-05-28,176.000,67.845,2022,148,362.0,2021
6357,2022-05-29,174.000,64.920,2022,149,363.0,2021
6358,2022-05-30,176.500,68.625,2022,150,364.0,2021
6359,2022-05-31,171.500,61.355,2022,151,365.0,2021


In [60]:
level[level["date"].dt.month == 11].groupby(level["date"].dt.year).max()

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
date,,,,,,,
2005,2005-11-30,478.000,1485.600,2005,334,183.0,2005
2006,2006-11-30,481.000,1505.700,2006,334,183.0,2006
2007,2007-11-30,491.667,1577.167,2007,334,183.0,2007
2008,2008-11-30,507.000,1523.800,2008,335,184.0,2008
2009,2009-11-30,489.000,1393.100,2009,334,183.0,2009
2010,2010-11-30,504.000,1501.600,2010,334,183.0,2010
2011,2011-11-30,462.000,1212.000,2011,334,183.0,2011
2012,2012-11-30,514.000,1576.400,2012,335,184.0,2012
2013,2013-11-30,494.667,1433.600,2013,334,183.0,2013


In [61]:
# either season
peaks = level.loc[level.groupby("seasonyear")["level"].idxmax()]
dummy_missing = pd.DataFrame(
    [{"seasonyear": 2022, "level": 0}, {"seasonyear": 2023, "level": 0}]
)
dummy_2024 = pd.DataFrame([{"seasonyear": 2024, "level": 670}])
peaks = pd.concat([peaks, dummy_missing, dummy_2024], ignore_index=True)
peaks["rank"] = peaks["level"].rank(ascending=False)
peaks["rp"] = len(peaks) / peaks["rank"]
peaks.sort_values("rank")

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear,rank,rp
15,2020-09-08,699.333,2656.667,2020.0,252.0,101.0,2020,1.0,20.000000
19,NaT,670.000,NaN,NaN,NaN,NaN,2024,2.0,10.000000
14,2019-09-01,636.000,2163.000,2019.0,244.0,93.0,2019,3.0,6.666667
7,2012-08-22,616.667,2478.667,2012.0,235.0,84.0,2012,4.0,5.000000
8,2013-08-30,611.750,2429.650,2013.0,242.0,91.0,2013,5.0,4.000000
11,2016-09-14,602.667,1922.667,2016.0,258.0,107.0,2016,6.0,3.333333
12,2017-09-11,595.667,1873.667,2017.0,254.0,103.0,2017,7.0,2.857143
13,2019-01-21,592.000,1848.000,2019.0,21.0,235.0,2018,8.0,2.500000
10,2015-08-05,579.667,2124.000,2015.0,217.0,66.0,2015,9.0,2.222222
5,2010-09-07,565.000,1993.000,2010.0,250.0,99.0,2010,10.0,2.000000


In [62]:
# crues locales
locale = level[level["dayofseason"] < FLOODSEASON_ENDDAY]
l_peaks = locale.loc[locale.groupby("seasonyear")["level"].idxmax()]
l_peaks[l_peaks["level"] >= THRESH]

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
2790,2012-08-22,616.667,2478.667,2012,235,84.0,2012
3163,2013-08-30,611.750,2429.650,2013,242,91.0,2013
4274,2016-09-14,602.667,1922.667,2016,258,107.0,2016
4636,2017-09-11,595.667,1873.667,2017,254,103.0,2017
4984,2018-08-25,581.000,1773.000,2018,237,86.0,2018
5356,2019-09-01,636.000,2163.000,2019,244,93.0,2019
5729,2020-09-08,699.333,2656.667,2020,252,101.0,2020


In [63]:
l_peaks

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
245,2005-09-03,497.000,1612.900,2005,246,95.0,2005
606,2006-08-30,512.000,1718.600,2006,242,91.0,2006
946,2007-08-05,465.000,1400.667,2007,217,66.0,2007
1397,2008-10-29,470.000,1264.000,2008,303,152.0,2008
1724,2009-09-21,491.667,1412.200,2009,264,113.0,2009
2075,2010-09-07,565.000,1993.000,2010,250,99.0,2010
2452,2011-09-19,462.667,1216.333,2011,262,111.0,2011
2790,2012-08-22,616.667,2478.667,2012,235,84.0,2012
3163,2013-08-30,611.750,2429.650,2013,242,91.0,2013
3589,2014-10-30,459.000,1192.600,2014,303,152.0,2014


In [64]:
l_triggers = locale[locale["level"] >= THRESH]
l_triggers = l_triggers.loc[
    l_triggers.groupby("seasonyear")["dayofseason"].idxmin()
]
l_triggers

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
2787,2012-08-19,602.833,2341.967,2012,232,81.0,2012
3163,2013-08-30,611.750,2429.650,2013,242,91.0,2013
4273,2016-09-13,585.000,1800.000,2016,257,106.0,2016
4632,2017-09-07,581.000,1773.000,2017,250,99.0,2017
4984,2018-08-25,581.000,1773.000,2018,237,86.0,2018
5351,2019-08-27,581.500,1776.500,2019,239,88.0,2019
5698,2020-08-08,583.458,1789.667,2020,221,70.0,2020


In [65]:
# crues guineennes
gui = level[level["dayofseason"] >= FLOODSEASON_ENDDAY]
g_peaks = gui.loc[gui.groupby("seasonyear")["level"].idxmax()]
g_peaks[g_peaks["level"] >= THRESH]

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
5133,2019-01-21,592.0,1848.0,2019,21,235.0,2018
5860,2021-01-17,582.0,1780.0,2021,17,231.0,2020


In [66]:
g_peaks

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
351,2005-12-18,489.000,1559.300,2005,352,201.0,2005
725,2006-12-27,501.000,1640.133,2006,361,210.0,2006
1096,2008-01-02,510.000,1704.333,2008,2,216.0,2007
1472,2009-01-12,528.667,1689.600,2009,12,226.0,2008
1837,2010-01-12,528.000,1684.400,2010,12,226.0,2009
2218,2011-01-28,538.000,1764.800,2011,28,242.0,2010
2543,2011-12-19,473.000,1284.100,2011,353,202.0,2011
2946,2013-01-25,537.000,1756.700,2013,25,239.0,2012
3287,2014-01-01,512.000,1561.200,2014,1,215.0,2013
3643,2014-12-23,509.667,1543.533,2014,357,206.0,2014


In [67]:
g_triggers = gui[gui["level"] >= THRESH]
g_triggers = g_triggers.loc[
    g_triggers.groupby("seasonyear")["dayofseason"].idxmin()
]
g_triggers

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
5115,2019-01-03,580.0,1766.0,2019,3,217.0,2018
5855,2021-01-12,580.0,1766.0,2021,12,226.0,2020


In [78]:
gui[(gui["seasonyear"] == 2018) & (gui["level"] >= 564)]

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
5092,2018-12-11,564.000,1660.000,2018,345,194.0,2018
5093,2018-12-12,565.000,1666.000,2018,346,195.0,2018
5094,2018-12-13,566.000,1673.000,2018,347,196.0,2018
5095,2018-12-14,567.000,1679.000,2018,348,197.0,2018
5096,2018-12-15,568.000,1686.000,2018,349,198.0,2018
...,...,...,...,...,...,...,...
5155,2019-02-12,575.000,1732.333,2019,43,257.0,2018
5156,2019-02-13,572.667,1716.667,2019,44,258.0,2018
5157,2019-02-14,570.000,1699.333,2019,45,259.0,2018
5158,2019-02-15,567.667,1683.667,2019,46,260.0,2018


In [79]:
gui[(gui["seasonyear"] == 2020) & (gui["level"] >= 564)]

,date,level,Discharges (m3/s),year,dayofyear,dayofseason,seasonyear
5829,2020-12-17,564.000,1660.000,2020,352,201.0,2020
5830,2020-12-18,564.000,1660.000,2020,353,202.0,2020
5831,2020-12-19,565.000,1666.000,2020,354,203.0,2020
5832,2020-12-20,565.000,1666.000,2020,355,204.0,2020
5833,2020-12-21,565.000,1666.000,2020,356,205.0,2020
...,...,...,...,...,...,...,...
5886,2021-02-12,569.727,1697.409,2021,43,257.0,2020
5887,2021-02-13,568.222,1687.667,2021,44,258.0,2020
5888,2021-02-14,568.261,1687.609,2021,45,259.0,2020
5889,2021-02-15,566.087,1673.348,2021,46,260.0,2020


In [112]:
nov_g_trigger = gui[
    (gui["seasonyear"].isin([2018, 2020])) & (gui["date"].dt.month == 11)
].copy()

In [113]:
nov_g_trigger["d_level"] = (
    nov_g_trigger["level"] - nov_g_trigger["level"].shift()
)

In [114]:
nov_g_trigger["d_level"] = nov_g_trigger["d_level"].apply(
    lambda x: x if x >= -1 else np.NaN
)

In [115]:
(580 - 564) / nov_g_trigger["d_level"].mean()

15.213114754098362